# Structured extraction benchmark — four models, one prompt

**Experimental. Does not modify the production document-processing pipeline.**

Takes the rotation-corrected Cloud Vision output in `extracted/final/`, groups it
into logical documents, and asks four models to extract the Draftly field schema
from identical page-tagged OCR text. Every returned value is then **grounded
deterministically against the Vision words** before it is allowed to count.

| Provider | Model | SDK |
|---|---|---|
| OpenAI | `gpt-5.6-terra` | `openai` |
| OpenAI | `gpt-5.6-luna` | `openai` |
| Google | `gemini-2.5-flash` | `google-genai` |
| Google | `gemini-2.5-flash-lite` | `google-genai` |

No LiteLLM. Each provider is called through its own official SDK; a thin wrapper
normalises the *result*, not the call.

## Verified before writing this notebook

All four models were probed for existence and parameter support rather than
assumed:

- All four are **present** and return schema-constrained JSON.
- **`temperature` is unsupported on both OpenAI models.** `temperature=0` returns
  `400 Unsupported value: 'temperature' does not support 0 with this model`. The
  Gemini models accept `temperature=0` and run deterministically.

That asymmetry is real and is **not** worked around. The OpenAI models run at
their default sampling, so their results carry run-to-run variance the Gemini
results do not. Small gaps between the two families should be read with that in
mind, and the repeat-stability cell measures it rather than assuming it away.

## Rules

- Credentials from environment variables only; never printed.
- Responses cached by `(provider, model, prompt hash, schema hash, document hash)`.
- **Gold values are never sent to a model.** They load only in the scoring cell,
  after every response is final.
- No client OCR text, extracted values, gold values or raw responses are printed.
  Outputs are counts, rates and timings.

In [ ]:
from __future__ import annotations

import hashlib, json, os, re, sys, time
from collections import Counter, defaultdict
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, Sequence

sys.path.insert(0, str(Path.cwd() if Path.cwd().name == "ocr-benchmark" else Path.cwd() / "ocr-benchmark"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

import config, normalize, viz

load_dotenv(config.ROOT / ".env", override=True)
viz.use_style()
pd.set_option("display.width", 180)

FINAL = Path(config.__file__).parent / "extracted" / "final"
CACHE = config.RENDERS / "llm-cache"
CACHE.mkdir(parents=True, exist_ok=True)

PROMPT_VERSION = "v1"

REGISTRY = json.loads((config.SCHEMAS / "registry-fields.json").read_text(encoding="utf-8"))["kinds"]
CRITICAL = set(json.loads((config.SCHEMAS / "critical-fields.json").read_text(encoding="utf-8"))["critical"])
FIELD_LABELS = {f["key"]: f["label"] for fs in REGISTRY.values() for f in fs}
FIELD_VALIDATOR = {f["key"]: f.get("validator") for fs in REGISTRY.values() for f in fs}

print(f"final/ pages: {len(list(FINAL.glob('*-p*.json')))}")
print(f"registry: {len(REGISTRY)} kinds, {len(FIELD_LABELS)} fields, {len(CRITICAL)} critical")

## Model registry and availability probe

Availability is checked live. An unreachable model is reported and **excluded**;
nothing is substituted for it.

In [ ]:
@dataclass(frozen=True)
class ModelSpec:
    provider: str
    model: str
    supports_temperature: bool
    # USD per 1M tokens. None = not verified -> cost reported as unknown, never invented.
    price_in: float | None = None
    price_out: float | None = None


MODELS = [
    ModelSpec("openai", "gpt-5.6-terra", supports_temperature=False),
    ModelSpec("openai", "gpt-5.6-luna", supports_temperature=False),
    ModelSpec("google", "gemini-2.5-flash", supports_temperature=True,
              price_in=0.30, price_out=2.50),
    ModelSpec("google", "gemini-2.5-flash-lite", supports_temperature=True,
              price_in=0.10, price_out=0.40),
]

_openai = None
_google = None


def openai_client():
    global _openai
    if _openai is None:
        from openai import OpenAI
        key = os.environ.get("OPENAI_API_KEY", "").strip()
        if not key:
            raise RuntimeError("OPENAI_API_KEY not set")
        _openai = OpenAI(api_key=key)
    return _openai


def google_client():
    global _google
    if _google is None:
        from google import genai
        key = os.environ.get("GEMINI_API_KEY", "").strip()
        if not key:
            raise RuntimeError("GEMINI_API_KEY not set")
        _google = genai.Client(api_key=key)
    return _google


def probe(spec):
    try:
        if spec.provider == "openai":
            ids = {m.id for m in openai_client().models.list()}
        else:
            ids = {m.name.replace("models/", "") for m in google_client().models.list()}
    except Exception as exc:
        return False, f"{type(exc).__name__}: {str(exc)[:100]}"
    return (True, "available") if spec.model in ids else (False, "not offered by this account")


AVAILABILITY = {}
for spec in MODELS:
    ok, why = probe(spec)
    AVAILABILITY[spec.model] = (ok, why)
    temp = "temperature ok" if spec.supports_temperature else "TEMPERATURE UNSUPPORTED"
    print(f"  {spec.provider:7s} {spec.model:24s} {'OK ' if ok else 'NO '} {why:32s} {temp}")

ACTIVE = [s for s in MODELS if AVAILABILITY[s.model][0]]
print(f"\n{len(ACTIVE)}/{len(MODELS)} active. Unavailable models are excluded, not replaced.")

## Logical documents from `final/`

Pages grouped by source document, concatenated with explicit page tags so a value
can be attributed to a page. Text comes from the **corrected** reading order, so
rotated pages read correctly.

In [ ]:
def page_text_in_reading_order(meta):
    words = sorted(meta["words"], key=lambda w: w.get("reading_order", 0))
    line, out, previous_y = [], [], None
    for word in words:
        top = min(p[1] for p in word["corrected_polygon"])
        if previous_y is not None and abs(top - previous_y) > 0.008:
            out.append(" ".join(line))
            line = []
        line.append(word["text"])
        previous_y = top
    if line:
        out.append(" ".join(line))
    return "\n".join(out)


documents = {}
for path in sorted(FINAL.glob("*-p*.json")):
    meta = json.loads(path.read_text(encoding="utf-8"))
    entry = documents.setdefault(meta["source_document"], {
        "source_document": meta["source_document"],
        "source_sha256": meta["source_sha256"],
        "pages": [],
    })
    entry["pages"].append(meta)

for entry in documents.values():
    entry["pages"].sort(key=lambda m: m["page_no"])
    entry["page_text"] = {m["page_no"]: page_text_in_reading_order(m) for m in entry["pages"]}
    entry["tagged_text"] = "\n\n".join(
        f"<<<PAGE {m['page_no']}>>>\n{entry['page_text'][m['page_no']]}" for m in entry["pages"])
    entry["doc_hash"] = hashlib.sha256(entry["tagged_text"].encode("utf-8")).hexdigest()[:16]

# The KIND only comes from expected-fields.json - never the field values.
kinds = {}
for case in config.case_dirs():
    path = config.expected_fields_path(case)
    if path.is_file():
        for name, payload in json.loads(path.read_text(encoding="utf-8")).items():
            kinds[name] = payload.get("kind", "other")

for name, entry in documents.items():
    entry["kind"] = kinds.get(name, "other")
    entry["field_keys"] = [f["key"] for f in REGISTRY.get(entry["kind"], [])]

display(pd.DataFrame([{
    "document": n[:42], "kind": e["kind"], "pages": len(e["pages"]),
    "chars": len(re.sub(r"\s+", "", e["tagged_text"])),
    "approx_tokens": len(e["tagged_text"]) // 4,
    "fields_requested": len(e["field_keys"]),
} for n, e in sorted(documents.items())]))

TARGETS = [e for e in documents.values() if e["field_keys"]]
print(f"{len(TARGETS)} documents with a known kind -> {len(TARGETS) * len(ACTIVE)} calls")

## Shared prompt and JSON schema

Identical for every model, generated from the registry. Every value is typed
`string`, never a number: `"0021"` decoding to `21` is the failure this whole
benchmark exists to catch.

In [ ]:
SYSTEM_PROMPT = (
    "You extract structured fields from OCR text of Sri Lankan land-registration "
    "documents. You return only what is present in the text. You never infer, "
    "translate, normalise or complete a value."
)

USER_TEMPLATE = """Extract the requested fields from the OCR text below.

Document type: {kind}

Requested fields:
{field_list}

Rules:
- Copy each value EXACTLY as it appears. Preserve leading zeros, punctuation and
  separators. Do not reformat dates, numbers or amounts.
- `evidence` must be a verbatim substring of the OCR text containing the value.
- `page` must be the page number from the <<<PAGE n>>> marker it came from.
- If a field is not present, omit it. Never guess or return a placeholder.
- `confidence` is your own 0-1 estimate that the value is correct.
- The text may contain Sinhala, Tamil and English. Do not translate any of it.

OCR text:
{text}
"""

RESULT_SCHEMA = {
    "type": "object",
    "properties": {
        "fields": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "key": {"type": "string"},
                    "value": {"type": "string"},
                    "page": {"type": "integer"},
                    "evidence": {"type": "string"},
                    "confidence": {"type": "number"},
                },
                "required": ["key", "value", "page", "evidence", "confidence"],
                "additionalProperties": False,
            },
        }
    },
    "required": ["fields"],
    "additionalProperties": False,
}

SCHEMA_HASH = hashlib.sha256(json.dumps(RESULT_SCHEMA, sort_keys=True).encode()).hexdigest()[:16]


def build_prompt(entry):
    field_list = "\n".join(
        f"  - {k}: {FIELD_LABELS.get(k, k)}" + ("  [critical]" if k in CRITICAL else "")
        for k in entry["field_keys"])
    return USER_TEMPLATE.format(kind=entry["kind"], field_list=field_list,
                                text=entry["tagged_text"])


def prompt_hash(prompt):
    return hashlib.sha256((SYSTEM_PROMPT + prompt + PROMPT_VERSION).encode()).hexdigest()[:16]


print(f"schema hash {SCHEMA_HASH} | prompt version {PROMPT_VERSION}")

## The shared wrapper

Two provider functions, one internal result type. It normalises the *result*; it
does not pretend the two APIs are the same call.

In [ ]:
@dataclass
class ModelResult:
    provider: str
    model: str
    document: str
    raw_fields: list = field(default_factory=list)
    prompt_tokens: int = 0
    completion_tokens: int = 0
    seconds: float = 0.0
    schema_valid: bool = True
    error: str = ""


MAX_OUTPUT_TOKENS = 8000


def _call_openai(spec, prompt):
    kwargs = {
        "model": spec.model,
        "messages": [{"role": "system", "content": SYSTEM_PROMPT},
                     {"role": "user", "content": prompt}],
        "response_format": {"type": "json_schema",
                            "json_schema": {"name": "draftly_fields",
                                            "schema": RESULT_SCHEMA, "strict": True}},
        "max_completion_tokens": MAX_OUTPUT_TOKENS,
    }
    # Omitted when unsupported rather than swallowed in a try/except: a silently
    # dropped parameter is how an unfair comparison hides.
    if spec.supports_temperature:
        kwargs["temperature"] = 0
    response = openai_client().chat.completions.create(**kwargs)
    usage = response.usage
    return (response.choices[0].message.content or "",
            usage.prompt_tokens, usage.completion_tokens)


def _call_google(spec, prompt):
    from google.genai import types
    kwargs = {
        "response_mime_type": "application/json",
        "response_json_schema": RESULT_SCHEMA,
        "system_instruction": SYSTEM_PROMPT,
        "max_output_tokens": MAX_OUTPUT_TOKENS,
    }
    if spec.supports_temperature:
        kwargs["temperature"] = 0
    response = google_client().models.generate_content(
        model=spec.model, contents=prompt,
        config=types.GenerateContentConfig(**kwargs))
    usage = response.usage_metadata
    return (response.text or "",
            int(usage.prompt_token_count or 0), int(usage.candidates_token_count or 0))


CALLERS = {"openai": _call_openai, "google": _call_google}


def cache_path(spec, entry, prompt, tag=""):
    name = f"{spec.model}__{prompt_hash(prompt)}__{SCHEMA_HASH}__{entry['doc_hash']}"
    if tag:
        name += f"__{tag}"
    return CACHE / spec.provider / (name.replace("/", "-") + ".json")


def run_model(spec, entry, attempt_tag=""):
    prompt = build_prompt(entry)
    dest = cache_path(spec, entry, prompt, attempt_tag)
    if dest.is_file():
        return ModelResult(**json.loads(dest.read_text(encoding="utf-8")))

    result = ModelResult(provider=spec.provider, model=spec.model,
                         document=entry["source_document"])
    began = time.time()
    try:
        text, prompt_tokens, completion_tokens = CALLERS[spec.provider](spec, prompt)
        result.prompt_tokens, result.completion_tokens = prompt_tokens, completion_tokens
        try:
            fields = json.loads(text).get("fields", [])
            if not isinstance(fields, list):
                raise ValueError("fields is not a list")
            result.raw_fields = [f for f in fields if isinstance(f, dict)]
        except Exception as exc:
            result.schema_valid = False
            result.error = f"schema: {type(exc).__name__}"
    except Exception as exc:
        result.error = f"{type(exc).__name__}: {str(exc)[:120]}"
        result.schema_valid = False
    result.seconds = round(time.time() - began, 2)

    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_text(json.dumps(asdict(result), ensure_ascii=False), encoding="utf-8")
    return result


print("wrapper ready for:", [s.model for s in ACTIVE])

In [ ]:
results = []
for spec in ACTIVE:
    for entry in TARGETS:
        result = run_model(spec, entry)
        results.append(result)
        flag = "" if result.schema_valid else "  [" + result.error + "]"
        print(f"  {spec.model:24s} {entry['source_document'][:32]:34s} "
              f"fields={len(result.raw_fields):3d} {result.seconds:6.1f}s{flag}")

print(f"\n{len(results)} calls | "
      f"{sum(r.prompt_tokens for r in results)} prompt tokens | "
      f"{sum(r.completion_tokens for r in results)} completion tokens")

## Deterministic grounding against Vision words

The part that makes an answer trustworthy. Each value is matched back to the
actual Vision words. A value not found in the OCR is **not accepted** — it is
flagged ungrounded, which is the invented-value signal.

Matching is substring containment on the shared normalizer, never fuzzy: fuzzy
matching would forgive exactly the near-miss identifier this is meant to catch.

In [ ]:
def word_index(entry):
    index = {}
    for meta in entry["pages"]:
        words = sorted(meta["words"], key=lambda w: w.get("reading_order", 0))
        index[meta["page_no"]] = [
            {"word_id": f"p{meta['page_no']}-w{i:04d}", "text": w["text"],
             "confidence": w["confidence"], "polygon": w["corrected_polygon"]}
            for i, w in enumerate(words)]
    return index


def locate(value, words):
    """Smallest run of consecutive words whose joined text contains the value."""
    needle = normalize.platform_norm(value)
    if not needle:
        return []
    for span in range(1, 9):
        for start in range(0, max(0, len(words) - span) + 1):
            window = words[start:start + span]
            joined = normalize.platform_norm("".join(w["text"] for w in window))
            if needle in joined:
                return window
    return []


def ground(result, entry):
    index = word_index(entry)
    page_norm = {p: normalize.platform_norm(t) for p, t in entry["page_text"].items()}
    out = []
    for raw in result.raw_fields:
        key = str(raw.get("key", "")).strip()
        value = normalize.null_collapse(raw.get("value"))
        page = raw.get("page")
        evidence = str(raw.get("evidence") or "")
        record = {
            "model": result.model, "document": entry["source_document"], "key": key,
            "value": value, "page": page, "critical": key in CRITICAL,
            "model_confidence": raw.get("confidence"),
            "in_schema": key in entry["field_keys"],
            "word_ids": [], "polygons": [],
            "value_grounded": False, "evidence_grounded": False,
            "page_correct": False, "validator_ok": None,
        }
        if value is None:
            out.append(record)
            continue

        if isinstance(page, int) and page in index:
            record["page_correct"] = normalize.platform_norm(value) in page_norm.get(page, "")
            span = locate(value, index[page])
            if span:
                record["value_grounded"] = True
                record["word_ids"] = [w["word_id"] for w in span]
                record["polygons"] = [w["polygon"] for w in span]
        # Fall back to any page, so a right value on a wrong page stays
        # distinguishable from an invented one.
        if not record["value_grounded"]:
            for page_no, words in index.items():
                span = locate(value, words)
                if span:
                    record["value_grounded"] = True
                    record["word_ids"] = [w["word_id"] for w in span]
                    record["polygons"] = [w["polygon"] for w in span]
                    break
        if evidence:
            record["evidence_grounded"] = any(
                normalize.platform_norm(evidence) in t for t in page_norm.values())

        validator = FIELD_VALIDATOR.get(key)
        if validator and validator in normalize.VALIDATORS:
            record["validator_ok"] = normalize.VALIDATORS[validator](value)
        out.append(record)
    return out


by_document = {e["source_document"]: e for e in TARGETS}
grounded = []
for result in results:
    entry = by_document.get(result.document)
    if entry:
        grounded.extend(ground(result, entry))

gdf = pd.DataFrame(grounded)
print(f"{len(gdf)} candidate fields")
if len(gdf):
    display(gdf.groupby("model").agg(
        candidates=("key", "size"),
        grounded=("value_grounded", "mean"),
        evidence_grounded=("evidence_grounded", "mean"),
        page_correct=("page_correct", "mean"),
        off_schema=("in_schema", lambda s: 1 - s.mean()),
    ).round(3))

## Scoring against the 37 gold fields

Gold values load **here for the first time**. They were never part of any prompt.
Exact match on the platform normalizer — no fuzzy matching on any field.

In [ ]:
gold = {}
for case in config.case_dirs():
    path = config.expected_fields_path(case)
    if path.is_file():
        for name, payload in json.loads(path.read_text(encoding="utf-8")).items():
            if payload.get("fields"):
                gold[name] = payload["fields"]

print(f"gold: {sum(len(v) for v in gold.values())} fields across {len(gold)} documents")

rows = []
for spec in ACTIVE:
    predicted = {(r["document"], r["key"]): r
                 for r in grounded if r["model"] == spec.model and r["value"] is not None}
    for document, fields in gold.items():
        for key, want in fields.items():
            record = predicted.get((document, key))
            value = record["value"] if record else None
            correct = (value is not None and normalize.platform_norm(str(value))
                       == normalize.platform_norm(str(want)))
            rows.append({"model": spec.model, "document": document, "key": key,
                         "critical": key in CRITICAL, "predicted": value is not None,
                         "correct": bool(correct),
                         "grounded": bool(record and record["value_grounded"]),
                         "extra": False})
    for (document, key), record in predicted.items():
        if key not in gold.get(document, {}):
            rows.append({"model": spec.model, "document": document, "key": key,
                         "critical": key in CRITICAL, "predicted": True, "correct": False,
                         "grounded": record["value_grounded"], "extra": True})

scored = pd.DataFrame(rows)
# Force real booleans. As object dtype, `~` performs bitwise negation (-1/-2)
# rather than boolean NOT, which silently breaks every filter below.
for column in ("predicted", "correct", "grounded", "critical", "extra"):
    scored[column] = scored[column].fillna(False).astype(bool)


def metrics_for(model):
    sub = scored[scored["model"] == model]
    gold_rows = sub[~sub["extra"]]
    tp = int(gold_rows["correct"].sum())
    predicted = int(sub["predicted"].sum())
    total_gold = len(gold_rows)
    precision = tp / predicted if predicted else 0.0
    recall = tp / total_gold if total_gold else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    crit = gold_rows[gold_rows["critical"]]
    model_results = [r for r in results if r.model == model]
    candidates = [g for g in grounded if g["model"] == model and g["value"] is not None]
    spec = next(s for s in ACTIVE if s.model == model)
    prompt_tokens = sum(r.prompt_tokens for r in model_results)
    completion_tokens = sum(r.completion_tokens for r in model_results)
    cost = (None if spec.price_in is None else
            round(prompt_tokens / 1e6 * spec.price_in
                  + completion_tokens / 1e6 * spec.price_out, 5))
    grounding = float(np.mean([c["value_grounded"] for c in candidates])) if candidates else None
    return {
        "model": model,
        "critical_exact": round(float(crit["correct"].mean()), 3) if len(crit) else None,
        "precision": round(precision, 3),
        "recall": round(recall, 3),
        "f1": round(f1, 3),
        "grounding_rate": None if grounding is None else round(grounding, 3),
        "ungrounded_rate": None if grounding is None else round(1 - grounding, 3),
        "invalid_schema_rate": round(float(np.mean([not r.schema_valid for r in model_results])), 3),
        "seconds_per_doc": round(float(np.mean([r.seconds for r in model_results])), 2),
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "est_usd": cost,
    }


table = pd.DataFrame([metrics_for(s.model) for s in ACTIVE])
display(table)
print("est_usd is None where per-token pricing was not verified; no figure is invented.")

In [ ]:
# Which gold fields sit on a page whose OCR text is predominantly Sinhala?
sinhala_fields = set()
for document, fields in gold.items():
    entry = by_document.get(document)
    if not entry:
        continue
    for page_no, text in entry["page_text"].items():
        si = len(re.findall(r"[\u0D80-\u0DFF]", text))
        latin = len(re.findall(r"[A-Za-z]", text))
        if si > latin:
            for key, want in fields.items():
                if normalize.platform_norm(str(want)) in normalize.platform_norm(text):
                    sinhala_fields.add((document, key))

total_gold = sum(len(v) for v in gold.values())
print(f"{len(sinhala_fields)} of {total_gold} gold fields appear on a Sinhala-dominant page")

if sinhala_fields:
    base = scored[~scored["extra"]].copy()
    base["sinhala"] = [(d, k) in sinhala_fields
                       for d, k in zip(base["document"], base["key"])]
    display(base.groupby(["model", "sinhala"])["correct"].agg(["mean", "size"]).round(3))
else:
    print("\nNOT MEASURABLE: no gold field resolves to a Sinhala-dominant page.")
    print("Every one of the 37 labels is a Latin/numeric identifier, so a model")
    print("that ignored all Sinhala text could still score well here. That is a")
    print("property of the label set, not evidence about the models.")

In [ ]:
if len(table):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
    x = np.arange(len(table))
    labels = [m.replace("gemini-", "g-").replace("gpt-", "") for m in table["model"]]

    ax = axes[0]
    ax.bar(x - 0.2, table["critical_exact"].fillna(0), 0.36, color=viz.SERIES[0], label="critical")
    ax.bar(x + 0.2, table["f1"], 0.36, color=viz.SERIES[1], label="F1 (all fields)")
    ax.set_xticks(x, labels, rotation=20, ha="right")
    ax.set_ylim(0, 1)
    ax.set_title("Accuracy")
    ax.legend()

    ax = axes[1]
    ax.bar(x - 0.2, table["grounding_rate"].fillna(0), 0.36,
           color=viz.STATUS["correct"], label="grounded")
    ax.bar(x + 0.2, table["ungrounded_rate"].fillna(0), 0.36,
           color=viz.STATUS["wrong"], label="ungrounded (invented)")
    ax.set_xticks(x, labels, rotation=20, ha="right")
    ax.set_ylim(0, 1)
    ax.set_title("Grounding — is the value actually in the OCR?")
    ax.legend()

    ax = axes[2]
    ax.bar(x, table["seconds_per_doc"], 0.55, color=viz.SEQ_HUE)
    for i, v in enumerate(table["seconds_per_doc"]):
        ax.text(i, v + 0.1, f"{v:.1f}s", ha="center", fontsize=9, color=viz.INK_2)
    ax.set_xticks(x, labels, rotation=20, ha="right")
    ax.set_title("Latency per document")

    plt.tight_layout()
    plt.show()

## Repeat stability

The OpenAI models cannot be pinned to `temperature=0`, so identical runs may
differ. Rather than assume the variance is negligible, this measures it: one
document, run twice, compared field by field.

In [ ]:
MEASURE_STABILITY = False

if not MEASURE_STABILITY:
    print("skipped — set MEASURE_STABILITY = True to repeat one document per model")
elif TARGETS:
    entry = max(TARGETS, key=lambda e: len(e["field_keys"]))
    print(f"repeating: {entry['source_document'][:40]}\n")
    for spec in ACTIVE:
        first = {f["key"]: str(f.get("value")) for f in run_model(spec, entry).raw_fields}
        second = {f["key"]: str(f.get("value"))
                  for f in run_model(spec, entry, attempt_tag="repeat").raw_fields}
        keys = set(first) | set(second)
        same = sum(1 for k in keys if first.get(k) == second.get(k))
        note = "" if spec.supports_temperature else "   (temperature unsupported)"
        print(f"  {spec.model:24s} {same}/{len(keys)} fields identical across two runs{note}")

## Reading the results

**Grounding is the load-bearing metric here**, more than accuracy. A model that
returns a plausible parcel number it invented is worse than one that returns
nothing, and the ungrounded rate catches that directly. Accuracy rests on 37 gold
fields; grounding is computed on every candidate produced.

### Known limits

- **The comparison is not perfectly matched.** The OpenAI models reject
  `temperature=0`, so they sample while Gemini is deterministic. The parameter is
  omitted rather than silently swallowed, and the stability cell measures the
  resulting variance.
- **37 gold fields on 4 documents.** Differences of a few points are noise.
- **Sinhala performance is likely unmeasurable.** Every gold label is a
  Latin/numeric identifier, so a model ignoring all Sinhala could still score
  well. Fixing that needs labels whose values are Sinhala words — names,
  addresses, boundaries.
- **Cost only where pricing was verified.** Otherwise null, not a guess.
- **OCR is the floor.** A field Vision never recognised cannot be extracted by
  any model, so these measure the model *given this OCR*, not in isolation.

Nothing here modifies the production pipeline.